In [ ]:
import sys
import os
import importlib.util
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
sys.path.insert(0, os.path.abspath('../'))
from src.env.env_wrapper import SingleAgentWrapper

DYNAMIC_LIB_ROOT = Path(os.getcwd()).absolute().parent / "multiagent-envs"
sys.path.append(str(DYNAMIC_LIB_ROOT))

def custom_load_scenario(scenario_name):
    path = DYNAMIC_LIB_ROOT / "multiagent" / "scenarios" / scenario_name
    if not os.path.exists(path):
        raise FileNotFoundError(f"Unable to locate scenario: {path}")
    
    spec = importlib.util.spec_from_file_location("custom_scenario", str(path))
    scenario_module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(scenario_module)
    return scenario_module.Scenario()

scenario_real = custom_load_scenario("simple_tag.py") 
world_real = scenario_real.make_world()

env = SingleAgentWrapper(world=world_real, scenario=scenario_real)

def draw_real_entity(ax, ent, shape_type="circle", align_physics=True):
    if shape_type == "circle":
        circle = plt.Circle(ent.state.p_pos, ent.size, color=ent.color, alpha=0.75, zorder=3)
        ax.add_patch(circle)
    elif shape_type == "rectangle":
        # The checkpoint has a side length of (0.05) but a collision domain of radius (0.1).
        # if align_physics is True, render the side length as (0.1)
        side_length = ent.size * 2 if align_physics else ent.size
        lower_left = ent.state.p_pos - side_length / 2
        rect = patches.Rectangle(lower_left, side_length, side_length, color=ent.color, alpha=0.95, zorder=2)
        ax.add_patch(rect)
        
        trigger_circle = plt.Circle(ent.state.p_pos, ent.size, fill=True, edgecolor=ent.color, linestyle=':', alpha=0.5, zorder=10)
        ax.add_patch(trigger_circle)
    
    ax.plot(*ent.state.p_pos, 'k+', markersize=4, zorder=4)

ALIGN_PHYSICS = True 

fig = plt.figure(figsize=(16, 8), dpi=400)
fig.suptitle(f"Environment Visualization", fontsize=16)

# ---------------- Left Side ----------------
grid_left = fig.add_gridspec(2, 2, left=0.05, right=0.55, wspace=0.2, hspace=0.2)
axs_snapshot = [fig.add_subplot(grid_left[i, j]) for i in range(2) for j in range(2)]

for idx, ax in enumerate(axs_snapshot):
    env.reset()
    
    rect = plt.Rectangle((-1, -1), 2, 2, fill=False, edgecolor='black', linewidth=1.5, linestyle='--')
    ax.add_patch(rect)

    real_prey = env.world.agents[1]
    real_pred = env.world.agents[0]
    real_check = env.world.check[0]
    real_landmarks = env.world.landmarks
    
    # Entities
    for l in real_landmarks: 
        draw_real_entity(ax, l, shape_type="circle", align_physics=ALIGN_PHYSICS)
    draw_real_entity(ax, real_check, shape_type="rectangle", align_physics=ALIGN_PHYSICS)
    draw_real_entity(ax, real_prey, shape_type="circle", align_physics=ALIGN_PHYSICS)
    draw_real_entity(ax, real_pred, shape_type="circle", align_physics=ALIGN_PHYSICS)
    
    # Distance aux lines
    ax.plot([real_prey.state.p_pos[0], real_check.state.p_pos[0]], 
            [real_prey.state.p_pos[1], real_check.state.p_pos[1]], 'g:', alpha=0.7, 
            label=f"Dist: {np.linalg.norm(real_prey.state.p_pos - real_check.state.p_pos):.2f}")
    ax.plot([real_prey.state.p_pos[0], real_pred.state.p_pos[0]], 
            [real_prey.state.p_pos[1], real_pred.state.p_pos[1]], 'r:', alpha=0.7, 
            label=f"Dist: {np.linalg.norm(real_prey.state.p_pos - real_pred.state.p_pos):.2f}")
    
    ax.set_title(f"Spawn #{idx+1}", fontsize=10)
    ax.set_xlim(-1.1, 1.1)
    ax.set_ylim(-1.1, 1.1)
    ax.set_aspect('equal')
    ax.grid(True, linestyle=':', alpha=0.5)

# ---------------- Right side ----------------
ax_stat = fig.add_subplot(fig.add_gridspec(1, 1, left=0.62, right=0.95)[0, 0])
rect = plt.Rectangle((-1, -1), 2, 2, fill=False, edgecolor='black', linewidth=2, linestyle='--')
ax_stat.add_patch(rect)

prey_hist, pred_hist, check_hist = [], [], []
statistical_samples = 1000

for _ in range(statistical_samples):
    env.reset() 
    prey_hist.append(env.world.agents[1].state.p_pos.copy())
    pred_hist.append(env.world.agents[0].state.p_pos.copy())
    check_hist.append(env.world.check[0].state.p_pos.copy())

prey_hist = np.array(prey_hist)
pred_hist = np.array(pred_hist)
check_hist = np.array(check_hist)

for l in env.world.landmarks: 
    draw_real_entity(ax_stat, l, shape_type="circle", align_physics=ALIGN_PHYSICS)
draw_real_entity(ax_stat, env.world.check[0], shape_type="rectangle", align_physics=ALIGN_PHYSICS)

# Draw Monte Carlo samples
ax_stat.scatter(prey_hist[:, 0], prey_hist[:, 1], c='#1F77B4', s=2, alpha=0.15, label='Prey')
ax_stat.scatter(pred_hist[:, 0], pred_hist[:, 1], c='#D62728', s=2, alpha=0.15, label='Predator')
ax_stat.scatter(check_hist[:, 0], check_hist[:, 1], c='#2CA02C', s=2, alpha=0.15, label='Checkpoint')

ax_stat.set_title(f"Monte Carlo Density Map", fontsize=12)
ax_stat.set_xlim(-1.1, 1.1)
ax_stat.set_ylim(-1.1, 1.1)
ax_stat.set_aspect('equal')
ax_stat.grid(True, linestyle=':', alpha=0.5)
ax_stat.legend(loc='lower left', markerscale=5)

plt.show()